# MR DRE — Kaggle (1 câu hoàn chỉnh / lượt)
Cùng mã nguồn local. Một câu phải có đủ R1, ba R2 và kết quả chấm trước khi xử lý câu kế tiếp.
Yêu cầu: bật **GPU** và **Internet**. Notebook không chứa bản sao mã nguồn nhúng.
Dùng bản ZIP của repo rework tải lên hoặc repository Git đã cập nhật lên đúng mã nguồn mới.


In [ ]:
# Sửa các giá trị ở đây trước khi chạy.
REPO_URL = "https://github.com/beta123ha/ReviFence.git"  # Chỉ dùng nếu đã push phiên bản mới.
SOURCE_ZIP = ""  # Đường dẫn ZIP repo đã upload; ưu tiên ZIP nếu được đặt.
RESUME_ZIP = ""  # Đường dẫn baseline-results.zip của phiên trước (nếu có).
REPO_DIR = "/kaggle/working/ReviFence-MRDRE"
OUTPUT_ROOT = "/kaggle/working/output"
CHECKPOINT_ZIP = "/kaggle/working/baseline-results.zip"
MODEL = "qwen3:4b-instruct"
COUNT = 5
FEEDBACK = "all"        # all | content | format | reflect
EXTRA_EVAL = "all"      # all = 6 chỉ số
BUDGET_MINUTES = 600
# Để tiếp tục, thêm ZIP checkpoint dưới dạng Kaggle Dataset rồi đặt RESUME_ZIP.


In [ ]:
import os, sys, json, glob, shutil, zipfile, subprocess
from pathlib import Path
target = Path(REPO_DIR)
if not target.exists() or not (target / "pyproject.toml").is_file():
    if SOURCE_ZIP:
        source = Path(SOURCE_ZIP)
        if not source.is_file():
            raise FileNotFoundError(source)
        with zipfile.ZipFile(source) as archive:
            names = archive.namelist()
            candidates = [p for p in names if p.endswith("/pyproject.toml")]
            if len(candidates) != 1:
                raise RuntimeError("ZIP phải chứa đúng một pyproject.toml của repo")
            prefix = candidates[0].removesuffix("pyproject.toml")
            target.mkdir(parents=True, exist_ok=True)
            for item in archive.infolist():
                name = item.filename
                if not name.startswith(prefix) or name.endswith("/"):
                    continue
                relative = Path(name[len(prefix):])
                if not relative.parts or ".." in relative.parts:
                    continue
                dest = target / relative
                dest.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as fin, dest.open("wb") as fout:
                    shutil.copyfileobj(fin, fout)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
if not (target / "src/mrdre_baseline/runtime/runner.py").is_file():
    raise RuntimeError("Repo được tải không phải phiên bản đã rework; hãy đặt SOURCE_ZIP hoặc push code mới")
if RESUME_ZIP:
    with zipfile.ZipFile(RESUME_ZIP) as archive:
        archive.extractall("/kaggle/working")
print("Source:", target, "Checkpoint restored:", bool(RESUME_ZIP))


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)


In [ ]:
subprocess.run([sys.executable, str(target / "scripts/bootstrap.py"),
                "--profile", "kaggle", "--workdir", "/kaggle/working",
                "--model", MODEL], cwd=target, check=True)


In [ ]:
command = [sys.executable, "-m", "mrdre_baseline.runtime.runner",
           "--count", str(COUNT), "--feedback", FEEDBACK,
           "--extra-eval", EXTRA_EVAL,
           "--output-root", OUTPUT_ROOT,
           "--checkpoint-zip", CHECKPOINT_ZIP,
           "--budget-minutes", str(BUDGET_MINUTES)]
result = subprocess.run(command, cwd=target, check=False)
print("Runner exit code:", result.returncode, "(0=complete, 3=budget, 2=interrupted)")


In [ ]:
subprocess.run([sys.executable, "-m", "mrdre_baseline.runtime.summary",
                "--output-root", OUTPUT_ROOT, "--feedback", FEEDBACK,
                "--extra-eval", EXTRA_EVAL], cwd=target, check=True)
print("Archive:", CHECKPOINT_ZIP)
